# Playground — Step 3: Concept-Pair Toolkit + ρ

New API:

- `Frame.rho(other)` — concept–concept correlation, handling frames of different ranks
  (zero-pad to common k; padding is rank-neutral)
- `fur.get_concept_cached(synset, min_lemmas, max_tokens)` — like `get_concept` but
  persists the frame tensor to `cache/concepts/`, keyed by model + synset + filter args + languages
- `fur.concept_pair(a, b, min_lemmas, max_tokens)` → `(Concept, Concept, rho)`

ρ is the interference predictor for E0: the hypothesis is that frame-averaging (F1.a)
degrades as ρ drops, while score-space composition (F2.a) should be less affected.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `frames` imports work

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.representations import FrameUnembeddingRepresentation

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

MIN_LEMMAS = 11
MAX_TOKENS = 3

## 1 — Sanity gate: ρ ordering on known pairs

`woman`/`man` (same semantic neighborhood) must correlate higher than `woman`/`dog`
(unrelated). If this fails, every downstream ρ-based conclusion is suspect.

In [ ]:
_, _, rho_close = fur.concept_pair("woman.n.01", "man.n.01", MIN_LEMMAS, MAX_TOKENS)
_, _, rho_far = fur.concept_pair("woman.n.01", "dog.n.01", MIN_LEMMAS, MAX_TOKENS)

print(f"rho(woman, man) = {rho_close:.4f}")
print(f"rho(woman, dog) = {rho_far:.4f}")

assert rho_close > rho_far, "rho ordering sanity check failed!"
print("sanity gate passed")

## 2 — Disk cache speedup

Second call loads the tensor from `cache/concepts/` instead of rebuilding from WordNet.

In [ ]:
start = time.perf_counter()
fur.get_concept_cached("joy.n.01", MIN_LEMMAS, MAX_TOKENS)
first = time.perf_counter() - start

start = time.perf_counter()
fur.get_concept_cached("joy.n.01", MIN_LEMMAS, MAX_TOKENS)
second = time.perf_counter() - start

print(f"first call:  {first * 1000:.1f} ms")
print(f"second call: {second * 1000:.1f} ms (from disk cache)")

## 3 — Which candidate synsets survive the filter?

A taste of the Step 8 regime taxonomy: check availability before assuming a pair exists.

In [ ]:
CANDIDATES = [
    "dog.n.01", "cat.n.01", "animal.n.01", "puppy.n.01",  # animals / hierarchy
    "man.n.01", "woman.n.01", "child.n.01", "girl.n.01", "boy.n.01",  # composition
    "joy.n.01", "sorrow.n.01", "happiness.n.01", "sadness.n.01",  # emotion antonyms
    "love.n.01", "hate.n.01", "war.n.01", "peace.n.01",  # abstract antonyms
    "music.n.01", "mathematics.n.01", "water.n.01", "food.n.01",  # unrelated domains
    "black.n.01", "white.n.01",  # paper bias pair
]

concepts = {}
for name in CANDIDATES:
    try:
        concepts[name] = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
    except ValueError:
        print("did not survive filter:", name)

print(f"\n{len(concepts)}/{len(CANDIDATES)} synsets available")

## 4 — ρ matrix over the survivors

In [ ]:
names = list(concepts)
rho = torch.zeros(len(names), len(names))
for i, a in enumerate(names):
    for j, b in enumerate(names):
        rho[i, j] = concepts[a].rho(concepts[b]).float().cpu()

matrix = pd.DataFrame(rho.numpy(), index=names, columns=names)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(matrix.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(names)), names, rotation=90)
ax.set_yticks(range(len(names)), names)
fig.colorbar(im)
plt.title("concept-concept rho")
plt.tight_layout()
plt.show()

### Things to look for (Step 8 preview)

- Are **antonyms** (`joy`/`sorrow`, `war`/`peace`, `black`/`white`) actually low-ρ,
  or geometrically similar as embedding-space folklore predicts?
- Are **unrelated domains** (`dog`/`mathematics`) the lowest stratum?
- Do **hypernym pairs** (`dog`/`animal`, `dog`/`puppy`) sit above coordinates (`dog`/`cat`)?

In [ ]:
pairs = [
    (a, b)
    for i, a in enumerate(names)
    for b in names[i + 1 :]
]
ranked = sorted(pairs, key=lambda p: -matrix.loc[p[0], p[1]])

print("highest-rho pairs:")
for a, b in ranked[:8]:
    print(f"  {matrix.loc[a, b]:+.3f}  {a} — {b}")
print("\nlowest-rho pairs:")
for a, b in ranked[-8:]:
    print(f"  {matrix.loc[a, b]:+.3f}  {a} — {b}")

## 5 — Composition teaser (RQ3 preview)

Does `mean(woman, child)` land near `girl` — *before any generation*?
Frames of different ranks are zero-padded to a common k before averaging
(the codebase's standard unequal-rank policy).

In [ ]:
from frames.linalg import Frame
from frames.linalg.orthogonalization import solve_procrustes


def padded_mean(*frames: Frame) -> Frame:
    k = max(len(f) for f in frames)
    padded = [torch.nn.functional.pad(f.tensor, (0, k - len(f))) for f in frames]
    return Frame(tensor=solve_procrustes(sum(padded)))


TRIPLES = [("woman.n.01", "child.n.01", "girl.n.01"), ("man.n.01", "child.n.01", "boy.n.01")]
for c1, c2, c3 in TRIPLES:
    if not all(c in concepts for c in (c1, c2, c3)):
        print(f"skip {c1} + {c2} ~ {c3} (concept unavailable)")
        continue
    mean = padded_mean(concepts[c1], concepts[c2])
    r_mean = mean.rho(concepts[c3]).item()
    r1 = concepts[c1].rho(concepts[c3]).item()
    r2 = concepts[c2].rho(concepts[c3]).item()
    print(
        f"{c1} + {c2} ~ {c3}:  rho(mean, C3) = {r_mean:+.3f}"
        f"   [rho(C1, C3) = {r1:+.3f}, rho(C2, C3) = {r2:+.3f}]"
    )